In [14]:
import pandas as pd
import os

# 1. 路径设置
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

raw_data_path = os.path.join("../", "raw_data", "Life expectancy at birth (years).csv")
output_file = os.path.join("./", "cleaned_gender_data.csv")

if os.path.exists(raw_data_path):
    df = pd.read_csv(raw_data_path)
    
    # 2. 基础过滤：只要寿命指标和明确的性别
    df_filtered = df[(df['IndicatorCode'] == 'WHOSIS_000001') & 
                     (df['Dim1'].isin(['Male', 'Female']))].copy()

    # 3. 提取数值 (处理 WHO 的 "74.2 [70.1-78.3]" 格式)
    df_filtered['Value'] = df_filtered['FactValueNumeric'].astype(float)

    # 4. 【核心步骤】按年份和性别计算全球平均值
    # 我们不按国家选，而是把所有国家在同一年、同一性别的数值加起来取平均
    global_avg = df_filtered.groupby(['Period', 'Dim1'])['Value'].mean().reset_index()

    # 5. 格式化：保留一位小数，并排序
    global_avg['Value'] = global_avg['Value'].round(1)
    global_avg = global_avg.sort_values(by=['Period', 'Dim1'])

    # 6. 保存
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    global_avg.to_csv(output_file, index=False)
    
    print("✅ 全球平均数据清洗完成！")
    print(f"数据量从 {len(df)} 行精简到了 {len(global_avg)} 行。")
    print(global_avg.head(10))
else:
    print("❌ 找不到原始文件")

✅ 全球平均数据清洗完成！
数据量从 24420 行精简到了 44 行。
   Period    Dim1  Value
0    2000  Female   69.5
1    2000    Male   64.6
2    2001  Female   69.8
3    2001    Male   64.9
4    2002  Female   70.0
5    2002    Male   65.2
6    2003  Female   70.2
7    2003    Male   65.5
8    2004  Female   70.5
9    2004    Male   65.8


In [16]:
import pandas as pd
import os

# 1. 设置路径 (建议统一使用 ./ 避免路径混乱)
raw_data_path = "../raw_data/Life expectancy at birth (years).csv"
output_path = "../data/cleaned_lifespan_map_data.csv"

def clean_lifespan_data(file_path):
    # 读取原始数据
    df = pd.read_csv(file_path, skipinitialspace=True)
    
    # 2. 筛选列
    cols_to_keep = [
        'Indicator',
        'SpatialDimValueCode', 
        'Location', 
        'Period', 
        'Dim1', 
        'FactValueNumeric'
    ]
    df = df[cols_to_keep]
    
    # 3. 【修正后的核心过滤】
    # 确保只保留“出生时预期寿命”且性别维度为“男女合计”
    df_clean = df[
        (df['Indicator'] == 'Life expectancy at birth (years)') & 
        (df['Dim1'] == 'Both sexes')
    ].copy()
    
    # 4. 数据类型转换
    df_clean['Period'] = df_clean['Period'].astype(int)
    # 将 FactValueNumeric 重命名为 Value 供 D3 使用
    df_clean['Value'] = df_clean['FactValueNumeric'].astype(float)
    
    # 5. 排序：按国家代码和年份排序
    df_clean = df_clean.sort_values(['SpatialDimValueCode', 'Period'])
    
    # 6. 只保留最终需要的列
    final_df = df_clean[['SpatialDimValueCode', 'Location', 'Period', 'Value']]
    
    # 保存
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final_df.to_csv(output_path, index=False)
    
    print(f"✅ 地图数据清洗完成！现在每个国家每年只有 1 条数据。")
    print(f"📊 预览：\n{final_df.head()}")

# 运行清洗
clean_lifespan_data(raw_data_path)

✅ 地图数据清洗完成！现在每个国家每年只有 1 条数据。
📊 预览：
      SpatialDimValueCode     Location  Period  Value
23394                 AFG  Afghanistan    2000  53.82
22285                 AFG  Afghanistan    2001  53.91
21183                 AFG  Afghanistan    2002  55.15
20079                 AFG  Afghanistan    2003  56.09
18964                 AFG  Afghanistan    2004  56.48
